# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

/var/folders/9p/f66yhymx35939zzlg51g0rrc0000gn/T/ipykernel_24022/3307975217.py:5: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 9
Very Important: Please Confirm the Iteration Number is Iteration 9
Very Important: Please Confirm the Iteration Number is Iteration 9


In [3]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1, n_trials=3)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  9

**************************************************************************************************************


[INFO 06-26 14:50:45] ax.service.ax_client: Generated new trial 27 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 0, 's2': 0, 's3': 0, 's4': 3, 's5': 0, 's6': 0, 's7': 0, 's8': 0, 'surfactant_conc': 1, 'drug_conc': 95} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:293: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))
[INFO 06-26 15:07:01] ax.service.ax_client: Generated new trial 28 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 0, 's2': 0, 's3': 0, 's4': 52, 's5': 0, 's6': 0, 's7': 0, 's8': 0, 'surfactant_conc': 1, 'drug_conc': 97} using model SAASBO.
/opt/anaconda3/envs/dr

Time taken for optimization: 54.19 mins
Time taken for optimization: 3251.3999999999996 seconds


# process results

In [4]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 48, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 10, 's8': 34, 'surfactant_conc': 85, 'drug_conc': 77})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 95, 's2': 21, 's3': 75, 's4': 63, 's5': 42, 's6': 71, 's7': 55, 's8': 99, 'surfactant_conc': 35, 'drug_conc': 2})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 52, 's2': 99, 's3': 4, 's4': 13, 's5': 22, 's6': 89, 's7': 28, 's8': 57, 'surfactant_conc': 18, 'drug_conc': 33})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', paramete

In [5]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [6]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: E7
Deep plate will start at: G7

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [7]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_9.py


In [8]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,0
1,1,0
2,2,0


In [9]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,initial_drug_conc_surfactant_conc_ratio,success,micelle_drug_conc,complexity
0,27,0,0,0,3,0,0,0,0,0.05,23.75,475.0,0,0.0,1
1,28,0,0,0,52,0,0,0,0,0.05,24.25,485.0,0,0.0,1
2,29,0,0,0,33,0,0,0,0,0.05,21.50,430.0,0,0.0,1


In [10]:
norm_results = hf.normalize_data(results, 'normalize')

In [11]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,initial_drug_conc_surfactant_conc_ratio,success,micelle_drug_conc,complexity
0,27,0,0,0,3,0,0,0,0,0.05,23.75,4.75,0.0,0.0,0.125
1,28,0,0,0,52,0,0,0,0,0.05,24.25,4.85,0.0,0.0,0.125
2,29,0,0,0,33,0,0,0,0,0.05,21.50,4.30,0.0,0.0,0.125


# load the results to the optimizer

In [12]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-26 15:56:57] ax.service.ax_client: Completed trial 27 with data: {'micelle_drug_conc': (0.0, None), 'success': (0.0, None), 'initial_drug_conc_surfactant_conc_ratio': (4.75, None), 'complexity': (0.125, None)}.
[INFO 06-26 15:56:57] ax.service.ax_client: Completed trial 28 with data: {'micelle_drug_conc': (0.0, None), 'success': (0.0, None), 'initial_drug_conc_surfactant_conc_ratio': (4.85, None), 'complexity': (0.125, None)}.
[INFO 06-26 15:56:57] ax.service.ax_client: Completed trial 29 with data: {'micelle_drug_conc': (0.0, None), 'success': (0.0, None), 'initial_drug_conc_surfactant_conc_ratio': (4.3, None), 'complexity': (0.125, None)}.


AxClient(experiment=Experiment(drug_surfactant))